### Install the dependencies

In [1]:
!pip install networkx matplotlib graphviz pygraphviz

  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached matplotlib-3.10.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached pygraphviz-1.14-cp311-cp311-linux_x86_64.whl
  Using cached contourpy-1.3.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.6/106.6 kB 1.5 MB/s eta 0:00:00a 0:00:01
  Using cached kiwisolver-1.4.8-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.2 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.7 MB/s eta 0:00:00
  Using cached pillow-11.2.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (8.9 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
Using cached networkx-3.5-py3-none-any.whl (2.0 MB)
Using cached matplotlib-3.10.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (8.6 MB)
   ━━━━━━━━━━━━

### Install spacy model

In [13]:
!pip install spacy
!python -m spacy download "en_core_web_trf"


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 2.0 MB/s eta 0:00:0000:0100:06

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


## GRAPHER

> Write the sentence you wish to graph in the pop up.

In [12]:
import spacy
import graphviz

nlp = spacy.load("en_core_web_trf")

def visualize_dependency_graph(text):
    """Saves a styled, graphical, top-down dependency tree image."""
    doc = nlp(text)
    dot = graphviz.Digraph(graph_attr={'rankdir': 'TB', 'nodesep': '1.5', 'ranksep': '1'})

    # Color scheme from your original script
    color_scheme = {
        "ROOT": "lightcoral",
        "attr": "#b0c4b1",
        "VERB": "lightgreen",
        "ADJ": "lightpink",
        "DET": "peachpuff",
        "PROPN": "plum",
        "NOUN": "plum",
    }
    default_color = "lightblue"

    for token in doc:
        if token.pos_ == 'PUNCT':
            continue

        # Determine the node's color
        # ROOT dependency color takes precedence
        if token.dep_ == "ROOT":
            color = color_scheme["ROOT"]
        else:
            color = color_scheme.get(token.pos_, default_color)
        
        # Recreate the detailed label from your original script
        label = f"({token.text})\nPOS: {token.pos_}\nTAG: {token.tag_}"

        # Add the styled node
        dot.node(
            name=str(token.i),
            label=label,
            style='filled',
            fillcolor=color,
            shape='circle',
            fixedsize='true',
            width='1.5'  # Diameter in inches
        )
        
        # Add the edge from its head
        if token.head.i != token.i:
            dot.edge(str(token.head.i), str(token.i), label=token.dep_)

    # Render and display the final tree
    dot.render('tree', format='svg', cleanup=True)
    print("✓ Styled dependency tree displayed.")

sentence = input("Write your sentence.") 
visualize_dependency_graph(sentence)

✓ Styled dependency tree displayed.


### Latex Generator

In [ ]:
import spacy
import math

# Load spaCy model
try:
    nlp = spacy.load("en_core_web_trf")
except OSError:
    print("Downloading en_core_web_trf model...")
    spacy.cli.download("en_core_web_trf")
    nlp = spacy.load("en_core_web_trf")

# --- TikZ Generation Configuration ---
# COLOR_MAP uses SVG color names, requiring [svgnames] option for xcolor.
# Capitalization matters for SVG names.
COLOR_MAP = {
    "ROOT_NODE_COLOR": "LightCoral",
    "VERB": "LightGreen",
    "ADJ": "LightPink",
    "DET": "PeachPuff",
    "PROPN": "Plum",
    "NOUN": "Plum",
    "ATTR_DEP_COLOR": "LimeGreen",
    "DEFAULT_NODE_COLOR": "LightBlue"
}

# LaTeX special characters that need escaping
LATEX_ESCAPE_CHARS = {
    '&': r'\&',
    '%': r'\%',
    '$': r'\$',
    '#': r'\#',
    '_': r'\_',
    '{': r'\{',
    '}': r'\}',
    '~': r'\textasciitilde{}',
    '^': r'\textasciicircum{}',
    '\\': r'\textbackslash{}',
}

def escape_latex(text):
    """Escapes special LaTeX characters in a string."""
    return "".join(LATEX_ESCAPE_CHARS.get(char, char) for char in text)

def get_node_color(token, is_root):
    """Determines the TikZ fill color for a token."""
    if is_root:
        return COLOR_MAP["ROOT_NODE_COLOR"]
    if token.dep_.lower() == "attr": # Check for 'attr' dependency
         return COLOR_MAP["ATTR_DEP_COLOR"]
    return COLOR_MAP.get(token.pos_, COLOR_MAP["DEFAULT_NODE_COLOR"])

# Global lists to store TikZ commands
node_definitions_tikz = []
edge_definitions_tikz = []
processed_tokens_for_nodes = set() # To avoid duplicate node definitions

def generate_tikz_recursive(doc, token_idx, parent_tikz_id=None, level=0, sibling_idx=0, num_siblings=0):
    """
    Recursively generates TikZ node and edge definitions.
    Uses token_idx (spaCy token.i) for tracking.
    """
    current_token = doc[token_idx]
    current_tikz_id = f"t{current_token.i}"

    if current_token.i in processed_tokens_for_nodes:
        if parent_tikz_id:
            dep_label = escape_latex(current_token.dep_)
            edge_label_opts = "auto" 
            if num_siblings > 0 :
                angle_approx = (sibling_idx - (num_siblings - 1) / 2.0)
                if abs(angle_approx) < 0.1 : 
                    edge_label_opts = "right"
                elif angle_approx < 0: 
                    edge_label_opts = "above"
                else: 
                    edge_label_opts = "above"
            edge_definitions_tikz.append(f"\\draw[edge style] ({parent_tikz_id}) -- ({current_tikz_id}) node[label style, {edge_label_opts}] {{{dep_label}}};")
        return

    processed_tokens_for_nodes.add(current_token.i)

    node_text_label = escape_latex(current_token.text)
    pos_label = escape_latex(current_token.pos_)
    tag_label = escape_latex(current_token.tag_)
    # Make the main token text Large and bold
    label_content = f"{{\\small\\textbf{{({node_text_label})}}}}\\\\POS: {pos_label}\\\\TAG: {tag_label}"
    
    is_root_node = current_token.dep_ == "ROOT"
    color = get_node_color(current_token, is_root_node)
    
    node_options_list = [f"node style, fill={color}"]

    x_offset_val_for_edge_label = 0 # Initialize for root or single child case

    if parent_tikz_id is None: 
        node_definitions_tikz.append(f"\\node ({current_tikz_id}) [{', '.join(node_options_list)}] {{{label_content}}};")
    else:
        y_distance = "1.8cm"  # Adjusted y_distance for potentially larger nodes
        if num_siblings == 1:
            position_keyword = "below"
            dist_str = y_distance
        else:
            mid_point = (num_siblings - 1) / 2.0
            horizontal_spacing_factor = 1.5  # Adjusted horizontal_spacing_factor
            x_offset_val_for_edge_label = (sibling_idx - mid_point) * horizontal_spacing_factor # Store for edge label
            
            if abs(x_offset_val_for_edge_label) < 0.1: 
                position_keyword = "below"
                dist_str = y_distance
            elif x_offset_val_for_edge_label < 0: 
                position_keyword = "below left"
                dist_str = f"{y_distance} and {abs(x_offset_val_for_edge_label):.1f}cm"
            else: 
                position_keyword = "below right"
                dist_str = f"{y_distance} and {abs(x_offset_val_for_edge_label):.1f}cm"
        
        node_options_list.append(f"{position_keyword}=of {parent_tikz_id}, node distance={dist_str}")
        node_definitions_tikz.append(f"\\node ({current_tikz_id}) [{', '.join(node_options_list)}] {{{label_content}}};")

        dep_label = escape_latex(current_token.dep_)
        edge_label_opts = "auto" 
        if abs(x_offset_val_for_edge_label) < 0.1 : edge_label_opts = "right"
        elif x_offset_val_for_edge_label < 0 : edge_label_opts = "sloped,above"
        else: edge_label_opts = "sloped,above"
        edge_definitions_tikz.append(f"\\draw[edge style] ({parent_tikz_id}) -- ({current_tikz_id}) node[label style, {edge_label_opts}] {{{dep_label}}};")

    children_indices = [c.i for c in current_token.children if c.pos_ != "PUNCT"]
    num_children = len(children_indices)
    for i, child_idx in enumerate(children_indices):
        generate_tikz_recursive(doc, child_idx, current_tikz_id, level + 1, i, num_children)


def spacy_to_tikz(text):
    global node_definitions_tikz, edge_definitions_tikz, processed_tokens_for_nodes
    node_definitions_tikz = []
    edge_definitions_tikz = []
    processed_tokens_for_nodes = set()

    doc = nlp(text)
    root_indices = [token.i for token in doc if token.dep_ == "ROOT" and token.pos_ != "PUNCT"]

    if not root_indices:
        first_token_idx = next((token.i for token in doc if token.pos_ != "PUNCT"), None)
        if first_token_idx is not None:
            root_indices = [first_token_idx]
        else:
            return "% No processable tokens found in text."

    generate_tikz_recursive(doc, root_indices[0])
    
    for token in doc:
        if token.pos_ != "PUNCT" and token.i not in processed_tokens_for_nodes:
            # Make the main token text Large and bold for unconnected nodes too
            temp_label = f"{{\\small\\textbf{{({escape_latex(token.text)})}}}}\\\\POS: {escape_latex(token.pos_)}\\\\TAG: {escape_latex(token.tag_)}" 
            temp_color = get_node_color(token, token.dep_ == "ROOT")
            unconnected_y_offset = len(doc) * 1.2 + len(processed_tokens_for_nodes) * 0.4 
            node_definitions_tikz.append(f"% Unconnected Node - manual placement might be needed:\n\\node (t{token.i}) [node style, fill={temp_color}, at={{(0,-{unconnected_y_offset}cm)}}] {{{temp_label}}};")
            processed_tokens_for_nodes.add(token.i)
            if token.head and token.head.i in processed_tokens_for_nodes and token.head.i != token.i:
                 dep_label = escape_latex(token.dep_)
                 edge_definitions_tikz.append(f"\\draw[edge style] (t{token.head.i}) -- (t{token.i}) node[label style, auto] {{{dep_label}}};")

    latex_preamble = f"""
\\begin{{tcolorbox}}[title=Example,
    boxrule=0.5pt,          % Thin rule
    sharp corners,          % Modern, sharp corners
    center,                 % Center the content (the figure)
]
\\begin{{figure}}[H]
\\begin{{center}} % Center the TikZ picture
\\scalebox{{.75}}{{
\\begin{{tikzpicture}}[
    node distance = 2cm and 1.5cm, % Adjusted node distance for larger nodes
    node style/.style = {{
        circle, 
        draw,
        thick,
        align=center,
        minimum size=2cm, % Increased minimum size for circles to fit Large text
        inner sep=1pt, 
        font=\\footnotesize % Smaller font for POS/TAG lines
    }},
	ex style/.style = {{
		draw=black!50,
		dashed,
		shorten >=1pt,
		shorten <=1pt
	}},
	ex arc/.style={{
		draw=black!50,
		dashed,
		shorten >=1pt,
		shorten <=1pt
	}},
    edge style/.style = {{
        draw,
        thick,
        -Latex 
    }},
    label style/.style = {{
        midway,
        font=\\small, % Edge labels
        inner sep=1pt 
    }}
]

% Node Definitions
"""
    
    nodes_str = "\n".join(node_definitions_tikz)
    edges_str = "\n".join(edge_definitions_tikz)
    
    latex_document = latex_preamble + nodes_str + "\n\n% Edge Definitions\n" + edges_str + \
                     "\n\n\\end{tikzpicture}}\n\\end{center} % End center environment\n\\end{figure}\n\\end{tcolorbox}"
    
    return latex_document

if __name__ == "__main__":
    example_text = input("Enter your statement: ")
    
    if example_text:
        tikz_output = spacy_to_tikz(example_text)
        print(tikz_output)
        
        # To save the output to a .tex file:
        # with open("dependency_tree.tex", "w") as f:
        #     f.write(tikz_output)
        # print("\nOutput saved to dependency_tree.tex")
    else:
        print("No input provided.")

